In [1]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Conv2D ,Flatten, MaxPooling2D, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import seaborn as sns
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import KFold
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import sys

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, BatchNormalization, ReLU,
    LSTM, Dense, Dropout
)


In [3]:
sensor_cols = ["Sensor_1", "Sensor_2", "Sensor_3", "Sensor_4"]

def make_windows(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[sensor_cols].iloc[i-window:i].values)
        y.append(data["RUL"].iloc[i])
    return np.array(X), np.array(y)


In [4]:
def build_cnn_lstm(input_shape, lstm_units):
    model = Sequential([
        Conv1D(32, 3, padding="same", input_shape=input_shape),
        BatchNormalization(),
        ReLU(),

        Conv1D(64, 3, padding="same"),
        BatchNormalization(),
        ReLU(),

        LSTM(lstm_units, return_sequences=False),

        Dense(64, activation="relu"),
        Dropout(0.3),

        Dense(1, activation="linear")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mae",
        metrics=["mae"]
    )
    return model


In [ ]:
df = pd.read_csv("bearing_train_final.csv")

WINDOW_LIST = [16, 32, 64]
LSTM_UNITS_LIST = [32, 64, 128]

results = []

for window in WINDOW_LIST:
    print(f"\n===== WINDOW = {window} =====")

    # 1) 윈도우 생성
    X, y = make_windows(df, window)

    # 2) 시간 기준 split
    split = int(len(X) * 0.8)
    X_train, X_val = X[:split], X[split:]
    y_train, y_val = y[:split], y[split:]

    # 3) 스케일링 (train 기준)
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, 4))

    def scale(X):
        return scaler.transform(X.reshape(-1, 4)).reshape(X.shape)

    X_train = scale(X_train)
    X_val   = scale(X_val)

    for units in LSTM_UNITS_LIST:
        print(f"  ▶ LSTM units = {units}")

        model = build_cnn_lstm(
            input_shape=X_train.shape[1:],
            lstm_units=units
        )

        es = EarlyStopping(
            monitor="val_mae",
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=200,
            batch_size=64,
            callbacks=[es],
            verbose=0
        )

        # 4) 평가
        y_val_pred = model.predict(X_val).squeeze()
        y_val_pred = np.clip(y_val_pred, 0, 1)

        mae = mean_absolute_error(y_val, y_val_pred)

        results.append({
            "WINDOW": window,
            "LSTM_units": units,
            "VAL_MAE": mae
        })

        print(f"     → VAL MAE = {mae:.6f}")



===== WINDOW = 16 =====
  ▶ LSTM units = 32


c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("VAL_MAE")

results_df